## Частина 1: Реалізація BPE з нуля

In [9]:
import re
from collections import defaultdict, Counter

def preprocess_text(text):
    text = text.lower()
    words = re.findall(r'\w+', text)
    return words

corpus = "low low low lower lower lowest the the the quick quick brown fox"
words = preprocess_text(corpus)
word_freqs = Counter(words)

def get_vocab(word_freqs):
    vocab = {}
    for word, freq in word_freqs.items():
        vocab[word] = list(word) + ['</w>']
    return vocab

def get_stats(vocab, word_freqs):
    pairs = defaultdict(int)
    for word, freq in word_freqs.items():
        symbols = vocab[word]
        for i in range(len(symbols)-1):
            pair = (symbols[i], symbols[i+1])
            pairs[pair] += freq
    return pairs

def merge_vocab(pair, vocab):
    new_vocab = {}
    pattern = ' '.join(pair)
    replacement = ''.join(pair)
    for word, tokens in vocab.items():
        tokens_str = ' '.join(tokens)
        tokens_str = tokens_str.replace(pattern, replacement)
        new_vocab[word] = tokens_str.split()
    return new_vocab

def encode_word_better(word, merges):
    tokens = list(word) + ['</w>']
    for pair in merges:
        i = 0
        new_tokens = []
        while i < len(tokens):
            if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == pair:
                new_tokens.append(''.join(pair))
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return tokens

class SimpleBPETokenizer:
    def __init__(self):
        self.merges = []
        self.vocab = set()

    def train(self, text, num_merges):
        words = preprocess_text(text)
        word_freqs = Counter(words)
        vocab = get_vocab(word_freqs)
        for tokens in vocab.values():
            self.vocab.update(tokens)
        for i in range(num_merges):
            pairs = get_stats(vocab, word_freqs)
            if not pairs:
                break
            best_pair = max(pairs, key=pairs.get)
            vocab = merge_vocab(best_pair, vocab)
            self.merges.append(best_pair)
            self.vocab.add(''.join(best_pair))

    def encode(self, text):
        words = preprocess_text(text)
        all_tokens = []
        for word in words:
            tokens = encode_word_better(word, self.merges)
            all_tokens.extend(tokens)
        return all_tokens

    def get_vocab_size(self):
        return len(self.vocab)

tokenizer = SimpleBPETokenizer()
tokenizer.train(corpus, num_merges=15)
print("Simple BPE Vocabulary size:", tokenizer.get_vocab_size())


Simple BPE Vocabulary size: 33


## Частина 2: Використання готових бібліотек (WordPiece та SentencePiece)

In [10]:
from transformers import BertTokenizer
import sentencepiece as spm
import os

bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
print("BERT vocab size:", bert_tokenizer.vocab_size)

training_text = """The quick brown fox jumps over the lazy dog.
Natural language processing is fascinating.
Tokenization is the first step in NLP.
Subword tokenization solves the OOV problem.
BPE, WordPiece, and SentencePiece are popular methods.
Machine learning models need tokenized input.
""" * 100

with open('train.txt', 'w') as f:
    f.write(training_text)

spm.SentencePieceTrainer.train(
    input='train.txt',
    model_prefix='sp_model',
    vocab_size=400, # 500
    model_type='bpe',
    character_coverage=1.0,
    pad_id=0, unk_id=1, bos_id=2, eos_id=3
)

sp = spm.SentencePieceProcessor()
sp.load('sp_model.model')
print("SentencePiece vocab size:", sp.vocab_size())


BERT vocab size: 30522
SentencePiece vocab size: 400


## Частина 3: Практичні завдання (Lab Exercises)

In [11]:
print("\n--- Exercise 1: Experiment with Vocabulary Size ---")
corpus_ex = corpus * 5 
for num_merges in [5, 20, 50]:
    tok = SimpleBPETokenizer()
    tok.train(corpus_ex, num_merges=num_merges)
    tokens = tok.encode("the quick brown fox")
    fertility = len(tokens) / 4
    print(f"Merges: {num_merges}, Vocab size: {tok.get_vocab_size()}, Fertility: {fertility:.2f}")
    
print("\n--- Exercise 2: Compare Segmentations ---")
word = "unhappiness"
print(f"Word: {word}")
print(f"Simple BPE: {tokenizer.encode(word)}")
print(f"BERT WordPiece: {bert_tokenizer.tokenize(word)}")
print(f"SentencePiece: {sp.encode_as_pieces(word)}")

print("\n--- Exercise 3: Handle Rare Words ---")
rare_words = [
    "pneumonoultramicroscopicsilicovolcanoconiosis", 
    "definately", 
    "ChatGPT", 
    "cryptocurrency"
]
for w in rare_words:
    bert_tokens = bert_tokenizer.tokenize(w)
    sp_tokens = sp.encode_as_pieces(w)
    print(f"{w}:\n  BERT: {bert_tokens} ({len(bert_tokens)} tokens)\n  SP: {sp_tokens} ({len(sp_tokens)} tokens)\n")

print("--- Exercise 4: Train Domain-Specific Tokenizer ---")
medical_text = "Azithromycin is a macrolide antibiotic used to treat pneumonia.\nDeoxyribonucleic acid stores genetic information in chromosomes.\n" * 50

with open('medical_train.txt', 'w', encoding='utf-8') as f:
    f.write(medical_text)

spm.SentencePieceTrainer.train(
    input='medical_train.txt',
    model_prefix='sp_med',
    vocab_size=40,
    model_type='bpe',
    character_coverage=1.0,
    pad_id=0, unk_id=1, bos_id=2, eos_id=3
)
sp_med = spm.SentencePieceProcessor()
sp_med.load('sp_med.model')

test_med = "Azithromycin is used to treat pneumonia."
print(f"Medical Test Sentence: {test_med}")
print(f"BERT General: {bert_tokenizer.tokenize(test_med)}")
print(f"Domain SP: {sp_med.encode_as_pieces(test_med)}")

spm.SentencePieceTrainer.train(
    input='medical_train.txt',
    model_prefix='sp_med',
    vocab_size=40,
    model_type='bpe',
    character_coverage=1.0,
    pad_id=0, unk_id=1, bos_id=2, eos_id=3
)
sp_med = spm.SentencePieceProcessor()
sp_med.load('sp_med.model')

test_med = "Azithromycin is used to treat pneumonia."
print(f"Medical Test Sentence: {test_med}")
print(f"BERT General: {bert_tokenizer.tokenize(test_med)}")
print(f"Domain SP: {sp_med.encode_as_pieces(test_med)}")



--- Exercise 1: Experiment with Vocabulary Size ---
Merges: 5, Vocab size: 23, Fertility: 4.50
Merges: 20, Vocab size: 38, Fertility: 2.00
Merges: 50, Vocab size: 43, Fertility: 1.00

--- Exercise 2: Compare Segmentations ---
Word: unhappiness
Simple BPE: ['u', 'n', 'h', 'a', 'p', 'p', 'i', 'n', 'e', 's', 's', '</w>']
BERT WordPiece: ['un', '##ha', '##pp', '##iness']
SentencePiece: ['▁', 'u', 'n', 'h', 'a', 'p', 'p', 'ine', 'ss']

--- Exercise 3: Handle Rare Words ---
pneumonoultramicroscopicsilicovolcanoconiosis:
  BERT: ['p', '##ne', '##um', '##ono', '##ult', '##ram', '##ic', '##ros', '##copic', '##sil', '##ico', '##vo', '##lc', '##ano', '##con', '##ios', '##is'] (17 tokens)
  SP: ['▁p', 'ne', 'u', 'mo', 'n', 'o', 'ul', 't', 'ra', 'm', 'ic', 'ro', 'sc', 'op', 'ic', 'si', 'l', 'ic', 'ov', 'ol', 'c', 'an', 'oc', 'on', 'io', 's', 'is'] (27 tokens)

definately:
  BERT: ['def', '##inate', '##ly'] (3 tokens)
  SP: ['▁', 'de', 'f', 'inat', 'el', 'y'] (6 tokens)

ChatGPT:
  BERT: ['chat', '